Exploratory Data Analysis (Statistical Foundation)

This notebook performs a comprehensive statistical exploratory data analysis (EDA) on Aadhaar enrolment, biometric update, and demographic update datasets for the period January–December 2025.

The objective of this EDA is to:

Understand statistical distributions and variability

Analyse geographic (state, district) and demographic (age) patterns

Derive metrics that support anomaly detection, efficiency analysis, and demand forecasting




In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [2]:
biometric = pd.read_csv("../data/processed/biometric_clean.csv")
demographic = pd.read_csv("../data/processed/demographic_clean.csv")
enrolment = pd.read_csv("../data/processed/enrolment_clean.csv")

In [3]:
print("Biometric shape:", biometric.shape)
print("Demographic shape:", demographic.shape)
print("Enrolment shape:", enrolment.shape)


biometric.head()

Biometric shape: (1765681, 6)
Demographic shape: (1597386, 6)
Enrolment shape: (982575, 7)


,date,state,district,pincode,bio_age_5_17,bio_age_17_
0,01-03-2025,Haryana,Mahendragarh,123029,280,577
1,01-03-2025,Bihar,Madhepura,852121,144,369
2,01-03-2025,Jammu & Kashmir,Punch,185101,643,1091
3,01-03-2025,Bihar,Bhojpur,802158,256,980
4,01-03-2025,Tamil Nadu,Madurai,625514,271,815


In [4]:
print("Biometric columns:")
print(biometric.columns.tolist())

print("\nDemographic columns:")
print(demographic.columns.tolist())

print("\nEnrolment columns:")
print(enrolment.columns.tolist())

Biometric columns:
['date', 'state', 'district', 'pincode', 'bio_age_5_17', 'bio_age_17_']

Demographic columns:
['date', 'state', 'district', 'pincode', 'demo_age_5_17', 'demo_age_17_']

Enrolment columns:
['date', 'state', 'district', 'pincode', 'age_0_5', 'age_5_17', 'age_18_greater']


## Variable Classification

Across the three datasets, the variables can be broadly classified as:

- **Geographic variables**: State, District
- **Temporal variables**: Month (Jan–Dec 2025)
- **Demographic variables**: Age / Age Group (where available)
- **Quantitative variables**: Count of enrolments or updates

The primary quantitative variable for analysis is the count field.

In [5]:
# Biometric total updates
biometric["bio_total"] = (
    biometric["bio_age_5_17"] +
    biometric["bio_age_17_"]
)

# Demographic total updates
demographic["demo_total"] = (
    demographic["demo_age_5_17"] +
    demographic["demo_age_17_"]
)

# Enrolment total
enrolment["enrol_total"] = (
    enrolment["age_0_5"] +
    enrolment["age_5_17"] +
    enrolment["age_18_greater"]
)

In [6]:
biometric["bio_total"].describe()

demographic["demo_total"].describe()

enrolment["enrol_total"].describe()

count    982575.000000
mean          5.425556
std          31.940087
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max        3965.000000
Name: enrol_total, dtype: float64

In [7]:
print("Biometric skewness:", stats.skew(biometric["bio_total"]))
print("Biometric kurtosis:", stats.kurtosis(biometric["bio_total"]))

print("Demographic skewness:", stats.skew(demographic["demo_total"]))
print("Enrolment skewness:", stats.skew(enrolment["enrol_total"]))

Biometric skewness: 16.49604331140372
Biometric kurtosis: 511.1587038118005
Demographic skewness: 36.12999318200907
Enrolment skewness: 38.71274578408205


## Distribution Characteristics

Aadhaar enrolment, biometric updates, and demographic updates exhibit strong right-skewness and high kurtosis.
This indicates highly uneven service demand, with a limited number of districts and states accounting for a large share of activity.

Such heavy-tailed behaviour justifies the use of robust statistics and anomaly detection techniques instead of normality-based assumptions.

In [8]:
# Biometric age split
biometric[["bio_age_5_17", "bio_age_17_"]].describe()

# Contribution share:
(
    biometric[["bio_age_5_17", "bio_age_17_"]].sum() /
    biometric["bio_total"].sum()
)

# Demographic age split
demographic[["demo_age_5_17", "demo_age_17_"]].describe()

# Enrolment age split
enrolment[["age_0_5", "age_5_17", "age_18_greater"]].describe()

,age_0_5,age_5_17,age_18_greater
count,982575.000000,982575.000000,982575.000000
mean,3.535489,1.720870,0.169197
std,17.730637,14.533385,3.252720
min,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,3.000000,1.000000,0.000000
max,2688.000000,1812.000000,855.000000


In [9]:
bio_state = biometric.groupby("state")["bio_total"].sum()
demo_state = demographic.groupby("state")["demo_total"].sum()
enrol_state = enrolment.groupby("state")["enrol_total"].sum()

bio_state.describe()

count    3.600000e+01
mean     1.896119e+06
std      2.389834e+06
min      4.745000e+03
25%      8.331850e+04
50%      1.113547e+06
75%      2.569855e+06
max      9.367083e+06
Name: bio_total, dtype: float64

In [10]:
bio_district = biometric.groupby("district")["bio_total"].sum()

bio_district.describe()

# Top districts:
bio_district.sort_values(ascending=False).head(10)

district
Pune          593592
Nashik        561893
Thane         559311
Jalgaon       408101
Ahmedabad     399202
Aurangabad    397283
Mumbai        395073
Ahmadnagar    353616
Jaipur        349532
Nagpur        342993
Name: bio_total, dtype: int64

In [11]:
state_metrics = pd.DataFrame({
    "enrolment": enrol_state,
    "biometric_updates": bio_state,
    "demographic_updates": demo_state
}).dropna()

# Efficiency ratio:
state_metrics["bio_to_enrol_ratio"] = (
    state_metrics["biometric_updates"] /
    state_metrics["enrolment"]
)

In [12]:
state_metrics.sort_values(
    "bio_to_enrol_ratio",
    ascending=False
).head(10)

,enrolment,biometric_updates,demographic_updates,bio_to_enrol_ratio
state,,,,
Andaman & Nicobar Islands,491,19988,5204,40.708758
Goa,2280,66554,27691,29.190351
Andhra Pradesh,124273,3610776,1642142,29.055193
Chandigarh,2620,73552,57395,28.073282
Tripura,11008,285324,104583,25.919695
Chhattisgarh,99773,2559111,1421369,25.649334
Maharashtra,363446,9020710,3824891,24.819946
Lakshadweep,199,4745,913,23.844221
Puducherry,2950,68485,24746,23.215254


In [13]:
state_metrics[[
    "enrolment",
    "biometric_updates",
    "demographic_updates"
]].corr(method="spearman")

,enrolment,biometric_updates,demographic_updates
enrolment,1.000000,0.934620,0.949292
biometric_updates,0.934620,1.000000,0.978121
demographic_updates,0.949292,0.978121,1.000000


In [14]:
# Extract month from date
biometric["date"] = pd.to_datetime(biometric["date"], dayfirst=True)
biometric["month"] = biometric["date"].dt.month

# Aggregate:
state_month = (
    biometric
    .groupby(["state", "month"])["bio_total"]
    .sum()
    .reset_index()
)

In [15]:
state_volatility = (
    state_month
    .groupby("state")["bio_total"]
    .std()
    /
    state_month
    .groupby("state")["bio_total"]
    .mean()
)

state_volatility.sort_values(ascending=False).head(10)

state
Mizoram           1.166890
Chandigarh        1.136973
Manipur           0.824561
Uttarakhand       0.557548
Sikkim            0.510306
Delhi             0.471885
Tripura           0.469718
Jharkhand         0.408967
Madhya Pradesh    0.400823
Gujarat           0.390693
Name: bio_total, dtype: float64

In [16]:
state_month["z_score"] = (
    state_month
    .groupby("state")["bio_total"]
    .transform(lambda x: (x - x.mean()) / x.std())
)

## Key EDA Takeaways

- Aadhaar service demand is highly skewed across states and districts.
- Biometric and demographic updates are dominated by the adult population (17+).
- Significant variation exists in biometric-to-enrolment ratios across states.
- Monthly volatility differs substantially by state, indicating uneven operational loads.
- These findings motivate anomaly detection and predictive modelling in subsequent steps.